# AtomicVision Judge Repro Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Adityabaskati-weeb/-AtomicVision-An-Autonomous-AI-Agent-for-Non-Destructive-Multi-Defect-Mapping/blob/main/notebooks/AtomicVision_Judge_Repro_Colab.ipynb)

This is the shareable, reproducible judge notebook for AtomicVision.

It runs the current held-out recovery path end to end:

1. clone the hardening branch,
2. generate the official `two_step_curriculum` SFT dataset,
3. validate with the NaN-safe trainer,
4. run a 5-update sanity train,
5. run the 40-update rebuild,
6. optionally publish the adapter to the Hugging Face Hub,
7. run strict + normalized held-out evaluation.

Recommended runtime: Colab T4 or better.

Optional secret: `HF_TOKEN` in Colab secrets if you want automatic adapter upload.

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Adityabaskati-weeb/-AtomicVision-An-Autonomous-AI-Agent-for-Non-Destructive-Multi-Defect-Mapping.git"
BRANCH = "main"
PROJECT_DIR = Path("/content/AtomicVision")

if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)
os.chdir(PROJECT_DIR)
print("cwd:", os.getcwd())
print("branch:", BRANCH)

In [ ]:
import subprocess
import sys
import torch

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "training/requirements-grpo.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openenv-core==0.2.3", "huggingface_hub"], check=True)

print("torch", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

In [ ]:
import os

try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
        print("HF_TOKEN loaded from Colab secrets.")
    else:
        print("HF_TOKEN not found in Colab secrets. Hub publish will be skipped.")
except Exception:
    print("google.colab.userdata unavailable. Hub publish will be skipped unless HF_TOKEN is already set.")

In [ ]:
from pathlib import Path

DATASET = PROJECT_DIR / "outputs" / "sft" / "atomicvision_two_step_curriculum_sft.jsonl"
VALIDATE_OUT = Path("/content/atomicvision-two-step-validate")
SANITY_OUT = Path("/content/atomicvision-format-submit-merged-sanity-lora")
FINAL_OUT = Path("/content/atomicvision-format-submit-merged-lora")
EVAL_JSON = PROJECT_DIR / "outputs" / "evaluation" / "atomicvision_judge_colab_eval.json"
HF_REPO_ID = "prodigyhuh/atomicvision-format-submit-merged-lora"

print(DATASET)
print(FINAL_OUT)
print(EVAL_JSON)

In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "training/generate_atomicvision_sft_data.py",
        "--profile", "two_step_curriculum",
        "--episodes-per-difficulty", "256",
        "--seed-start", "1000",
        "--difficulties", "medium", "hard",
        "--min-scan-improvement", "0.25",
        "--output-jsonl", str(DATASET),
    ],
    check=True,
)

In [ ]:
import shutil
import subprocess
import sys

if VALIDATE_OUT.exists():
    shutil.rmtree(VALIDATE_OUT)

subprocess.run(
    [
        sys.executable,
        "training/train_sft_atomicvision_safe.py",
        "--dataset-jsonl", str(DATASET),
        "--output-dir", str(VALIDATE_OUT),
        "--validate-only",
    ],
    check=True,
)

In [ ]:
import shutil
import subprocess
import sys

if SANITY_OUT.exists():
    shutil.rmtree(SANITY_OUT)

subprocess.run(
    [
        sys.executable,
        "training/train_sft_atomicvision_safe.py",
        "--dataset-jsonl", str(DATASET),
        "--model", "Qwen/Qwen3-1.7B",
        "--output-dir", str(SANITY_OUT),
        "--max-examples", "64",
        "--max-updates", "5",
        "--grad-accum", "8",
        "--max-length", "1536",
        "--learning-rate", "1e-5",
        "--max-grad-norm", "1.0",
        "--checkpoint-steps", "5",
        "--overwrite-output-dir",
    ],
    check=True,
)

In [ ]:
import shutil
import subprocess
import sys

if FINAL_OUT.exists():
    shutil.rmtree(FINAL_OUT)

subprocess.run(
    [
        sys.executable,
        "training/train_sft_atomicvision_safe.py",
        "--dataset-jsonl", str(DATASET),
        "--model", "Qwen/Qwen3-1.7B",
        "--output-dir", str(FINAL_OUT),
        "--max-updates", "40",
        "--grad-accum", "8",
        "--max-length", "1536",
        "--learning-rate", "1e-5",
        "--max-grad-norm", "1.0",
        "--checkpoint-steps", "20", "40",
        "--overwrite-output-dir",
    ],
    check=True,
)

print("checkpoint-20:", FINAL_OUT / "checkpoint-20")
print("checkpoint-40:", FINAL_OUT / "checkpoint-40")

In [ ]:
import os
import subprocess
import sys

if os.environ.get("HF_TOKEN"):
    subprocess.run(
        [
            sys.executable,
            "training/publish_adapter_to_hub.py",
            "--adapter-dir", str(FINAL_OUT),
            "--repo-id", HF_REPO_ID,
            "--base-model", "Qwen/Qwen3-1.7B",
            "--include-zip",
        ],
        check=True,
    )
    print("HF publish done:", f"https://huggingface.co/{HF_REPO_ID}")
else:
    print("HF_TOKEN not set, so Hub publish was skipped.")

In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "training/evaluate_atomicvision_adapter.py",
        "--adapter-dir", str(FINAL_OUT),
        "--base-model", "Qwen/Qwen3-1.7B",
        "--difficulties", "medium", "hard",
        "--episodes", "32",
        "--seed-start", "1000",
        "--modes", "strict", "normalized",
        "--output-json", str(EVAL_JSON),
    ],
    check=True,
)

In [ ]:
import json
from pathlib import Path

report = json.loads(Path(EVAL_JSON).read_text(encoding="utf-8"))
print("HELD-OUT EVAL (HONEST GATE)")
print("| difficulty | policy | reward | outcome | penalty | f1 | mae | steps | cost | fail | done | strict | normalized | first_valid | first_prior | submit |")
print("| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |")

for difficulty, result in report["results"].items():
    for key, label in [
        ("baseline_prior_submit", "baseline"),
        ("strict_adapter", "strict_adapter"),
        ("normalized_adapter", "normalized_adapter"),
    ]:
        row = result[key]
        print(
            f"| {difficulty} | {label} | "
            f"{row['mean_reward']:.4f} | {row['mean_outcome_reward_total']:.4f} | {row['mean_penalty_total']:.4f} | "
            f"{row['mean_f1']:.4f} | {row['mean_mae']:.5f} | "
            f"{row['mean_steps']:.2f} | {row['mean_scan_cost']:.2f} | "
            f"{row['tool_failure_rate']:.3f} | {row['done_rate']:.3f} | "
            f"{row['strict_tool_call_pass_rate']:.2f} | {row['normalized_tool_call_pass_rate']:.2f} | "
            f"{row['first_action_valid_rate']:.2f} | {row['first_action_ask_prior_rate']:.2f} | "
            f"{row['submit_action_rate']:.2f} |"
        )

print("\nSaved JSON:", EVAL_JSON)